# FewShot TopUp To20 — CRC-100K Patches to Exact 20

**Goal:** For classes shared between CRC-100K and the pathologist set
(ADI, DEB, MUC, NORM, MUS, STR), add enough CRC-100K patches so that
`pathologist_count + crc_to_sample = 20` for each class, then append those
rows to the 101-row pathologist Excel → `fewshot_combined_to20.xlsx`.

**LYM** is also included as a CRC-100K top-up class: 0 pathologist rows → 20 CRC-sampled patches.

**Classes left untouched (no CRC-100K equivalent):** PLC, MES, SIG, ADE, CAR, PDC, LYA.  
**LYA and LYM are distinct — LYA = lymphoid aggregates (pathologist), LYM = lymphocytes (CRC-100K).**  
**BACK is dropped from the combined Excel entirely.**  
**No existing file is modified.**

In [1]:
import os
import random as _rnd
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import sys
sys.argv = ['']

from conch.open_clip_custom import create_model_from_pretrained

print('Imports OK')

Imports OK


c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# ── Inputs (read-only) ─────────────────────────────────────────────────────
CRC_CACHE   = 'Results/Few_shot_features/conch_train_features.pt'
EXCEL_FILE  = 'fewshot_101_patches.xlsx'
CRC_DIR     = 'D:/Aamir Gulzar/dataset/CRC100K/NCT-CRC-HE-100K-NONORM'
PATCH_DIR   = 'D:/Aamir Gulzar/KSA_project2/dataset/patch_data'
CONCH_CKPT  = 'checkpoints/conch/pytorch_model.bin'

# ── Outputs (new files only) ────────────────────────────────────────────────
TOPUP_CACHE = 'Results/Few_shot_features/conch_crc_topup_features.pt'
EXCEL_OUT   = 'fewshot_combined_to20.xlsx'

# ── Top-up parameters ───────────────────────────────────────────────────────
# LYM has 0 pathologist rows → samples all 20 from CRC-100K (LYA is a separate class, untouched)
TOPUP_CLASSES = ['ADI', 'DEB', 'MUC', 'NORM', 'MUS', 'STR', 'LYM']
NO_TOPUP      = ['PLC', 'MES', 'SIG', 'ADE', 'CAR', 'PDC', 'LYA']  # untouched
TARGET        = 20      # each top-up class reaches exactly this count in the Excel
RANDOM_STATE  = 42
EXTENSIONS    = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
os.makedirs('Results/Few_shot_features', exist_ok=True)
print('Config ready.')

Device: cuda:0
Config ready.


In [3]:
print('Loading CONCH model...')
model, preprocess = create_model_from_pretrained(
    model_cfg='conch_ViT-B-16',
    checkpoint_path=CONCH_CKPT,
    device=DEVICE,
    force_image_size=224,
)
_ = model.eval()
print('CONCH model loaded.')

Loading CONCH model...
CONCH model loaded.


## Step 1 — Compute Top-Up Needs

For each top-up class: `need = TARGET - pathologist_count`.  
The CRC feature cache is loaded here for reference (used in verification) but
does **not** influence how many patches are sampled — only the pathologist count does.  
Print the full table **before** any sampling.

In [4]:
print('=' * 65)
print('STEP 1 -- Compute top-up needs')
print('=' * 65)

# CRC cache loaded for reference / downstream verification only
crc_cache   = torch.load(CRC_CACHE, weights_only=False)
y_crc_full  = crc_cache['labels']
crc_counts  = {v: int(c) for v, c in
               zip(*np.unique(y_crc_full, return_counts=True))}
print(f'  CRC cache: {len(y_crc_full):,} total features, {len(crc_counts)} classes')

# Pathologist Excel counts
df_orig     = pd.read_excel(EXCEL_FILE)
path_counts = df_orig['Primary Label'].value_counts().to_dict()
orig_cols   = df_orig.columns.tolist()
print(f'  Pathologist Excel: {len(df_orig)} rows, {len(path_counts)} classes')

# need = TARGET - pathologist_count  (no minimum buffer)
rows = []
for cls in TOPUP_CLASSES:
    path_n = int(path_counts.get(cls, 0))
    need   = TARGET - path_n
    rows.append({
        'label':             cls,
        'pathologist_count': path_n,
        'crc_to_sample':     need,
        'final_total':       path_n + need,
    })

needs_df = pd.DataFrame(rows)

print()
print(needs_df.to_string(index=False))
print(f'  Total new CRC patches to sample: {int(needs_df["crc_to_sample"].sum())}')

STEP 1 -- Compute top-up needs
  CRC cache: 100,000 total features, 9 classes
  Pathologist Excel: 101 rows, 13 classes

label  pathologist_count  crc_to_sample  final_total
  ADI                  5             15           20
  DEB                  5             15           20
  MUC                  5             15           20
 NORM                  5             15           20
  MUS                 10             10           20
  STR                 11              9           20
  LYM                  0             20           20
  Total new CRC patches to sample: 99


## Step 2 — Sample New Patches from Raw CRC-100K Directory

In [5]:
print('=' * 65)
print('STEP 2 -- Sample new patches from raw CRC-100K directory')
print('=' * 65)

if not os.path.isdir(CRC_DIR):
    raise FileNotFoundError(f'CRC-100K directory not found: {CRC_DIR}')

new_patch_records = []   # {label, filename, full_path}

for _, row in needs_df.iterrows():
    cls  = row['label']
    need = int(row['crc_to_sample'])
    cls_dir = os.path.join(CRC_DIR, cls)

    if not os.path.isdir(cls_dir):
        print(f'  [WARN] Folder not found: {cls_dir}')
        continue

    all_files = sorted(
        f for f in os.listdir(cls_dir)
        if Path(f).suffix.lower() in EXTENSIONS
    )

    rng_local = _rnd.Random(RANDOM_STATE)
    rng_local.shuffle(all_files)
    sampled = all_files[:need]

    for fname in sampled:
        new_patch_records.append({
            'label':     cls,
            'filename':  fname,
            'full_path': os.path.join(cls_dir, fname),
        })
    print(f'  {cls}: sampled {len(sampled)} of {need} needed (from {len(all_files):,} files)')

df_new_patches = pd.DataFrame(new_patch_records)
print(f'  Total new patches: {len(df_new_patches)}')
print(df_new_patches['label'].value_counts().sort_index().to_string())

STEP 2 -- Sample new patches from raw CRC-100K directory
  ADI: sampled 15 of 15 needed (from 10,407 files)
  DEB: sampled 15 of 15 needed (from 11,512 files)
  MUC: sampled 15 of 15 needed (from 8,896 files)
  NORM: sampled 15 of 15 needed (from 8,763 files)
  MUS: sampled 10 of 10 needed (from 13,536 files)
  STR: sampled 9 of 9 needed (from 10,446 files)
  LYM: sampled 20 of 20 needed (from 11,557 files)
  Total new patches: 99
label
ADI     15
DEB     15
LYM     20
MUC     15
MUS     10
NORM    15
STR      9


## Step 3 — Extract CONCH Features for New CRC Patches

Saves to `Results/Few_shot_features/conch_crc_topup_features.pt`.  
Delete that file to force re-extraction on re-run.

In [6]:
print('=' * 65)
print('STEP 3 -- Extract CONCH features for new CRC patches')
print('=' * 65)

expected_n = len(df_new_patches)

# Invalidate stale cache if it was built with a different patch count
if os.path.exists(TOPUP_CACHE):
    tc_peek = torch.load(TOPUP_CACHE, weights_only=False)
    cached_n = len(tc_peek['labels'])
    if cached_n != expected_n:
        print(f'  [INFO] Stale cache: {cached_n} features but {expected_n} patches expected.')
        print('  Deleting stale cache and re-extracting...')
        del tc_peek
        os.remove(TOPUP_CACHE)

if os.path.exists(TOPUP_CACHE):
    print(f'  Cache exists with correct size ({expected_n}). Loading...')
    tc      = torch.load(TOPUP_CACHE, weights_only=False)
    X_topup = tc['features']
    y_topup = tc['labels']
    print(f'  Loaded {X_topup.shape[0]} cached features.')
else:
    print(f'  Extracting {expected_n} patch features...')
    features, labels, failed = [], [], []
    with torch.inference_mode():
        for _, row in tqdm(df_new_patches.iterrows(),
                           total=len(df_new_patches), desc='TopUp patches'):
            try:
                img  = Image.open(row['full_path']).convert('RGB')
                t    = preprocess(img).unsqueeze(0).to(DEVICE)
                feat = model.encode_image(t)
                feat = feat / feat.norm(dim=-1, keepdim=True)
                features.append(feat.cpu().squeeze(0))
                labels.append(row['label'])
            except Exception as e:
                failed.append(row['filename'])
                print(f'  [WARN] {row["full_path"]}: {e}')

    X_topup = torch.stack(features)
    y_topup = np.array(labels)
    torch.save({'features': X_topup, 'labels': y_topup}, TOPUP_CACHE)
    print(f'  Extracted {X_topup.shape[0]} features.')
    if failed:
        print(f'  Failed ({len(failed)}): {failed}')
    print(f'  Saved -> {TOPUP_CACHE}')

print(f'  Top-up feature shape: {X_topup.shape}')
vals, cnts = np.unique(y_topup, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'    {v}: {c}')

STEP 3 -- Extract CONCH features for new CRC patches
  [INFO] Stale cache: 79 features but 99 patches expected.
  Deleting stale cache and re-extracting...
  Extracting 99 patch features...


TopUp patches:   0%|          | 0/99 [00:00<?, ?it/s]c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\layers\attention.py:80: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(
TopUp patches: 100%|██████████| 99/99 [00:01<00:00, 65.09it/s]

  Extracted 99 features.
  Saved -> Results/Few_shot_features/conch_crc_topup_features.pt
  Top-up feature shape: torch.Size([99, 512])
    ADI: 15
    DEB: 15
    LYM: 20
    MUC: 15
    MUS: 10
    NORM: 15
    STR: 9


## Step 4 — Build fewshot_combined_to20.xlsx

Original 101 rows are copied verbatim (unchanged).  
New CRC rows use the same column schema plus a `Full Path` column.  
BACK is dropped if present. `#` is re-indexed sequentially.

In [8]:
print('=' * 65)
print('STEP 4 -- Build fewshot_combined_to20.xlsx')
print('=' * 65)

# ── Original rows: add Full Path, keep everything else untouched ─────────────
df_orig_out = df_orig.copy()
df_orig_out['Full Path'] = df_orig_out['Filename'].apply(
    lambda f: os.path.join(PATCH_DIR, str(f)) if pd.notna(f) else ''
)

all_cols = orig_cols + ['Full Path']

# ── New CRC top-up rows ───────────────────────────────────────────────────────
new_rows = []
for i, rec in enumerate(new_patch_records):
    rd = {c: np.nan for c in all_cols}
    rd['#']             = len(df_orig) + i + 1
    rd['Filename']      = rec['filename']
    rd['Source Sheet']  = 'CRC100K TopUp'
    rd['True Label']    = rec['label']
    rd['Final Label']   = rec['label']
    rd['Primary Label'] = rec['label']
    rd['Full Path']     = rec['full_path']
    new_rows.append(rd)

df_new_rows = pd.DataFrame(new_rows, columns=all_cols)

# ── Concatenate ───────────────────────────────────────────────────────────────
df_combined = pd.concat([df_orig_out, df_new_rows], ignore_index=True)

# ── Drop BACK ────────────────────────────────────────────────────────────────
back_before  = int((df_combined['Primary Label'] == 'BACK').sum())
df_combined  = df_combined[
    df_combined['Primary Label'] != 'BACK'
].reset_index(drop=True)
if back_before:
    print(f'  Dropped {back_before} BACK row(s).')
else:
    print('  No BACK rows present (as expected).')

# ── Re-index # ───────────────────────────────────────────────────────────────
df_combined['#'] = range(1, len(df_combined) + 1)

# ── Save ─────────────────────────────────────────────────────────────────────
df_combined.to_excel(EXCEL_OUT, index=False)
print(f'  Saved {len(df_combined)} rows -> {EXCEL_OUT}')
print()
print('  Per-label counts in new Excel:')
print(df_combined['Primary Label'].value_counts().sort_index().to_string())

STEP 4 -- Build fewshot_combined_to20.xlsx
  No BACK rows present (as expected).
  Saved 200 rows -> fewshot_combined_to20.xlsx

  Per-label counts in new Excel:
Primary Label
ADE     11
ADI     20
CAR     11
DEB     20
LYA     12
LYM     20
MES      8
MUC     20
MUS     20
NORM    20
PDC     10
PLC      2
SIG      6
STR     20


## Step 5 — Verification

Checks: original files unchanged · zero BACK · zero LYM-LYA confusion ·
correct row count · CRC cache untouched. Ends with the final summary table.

In [9]:
print('=' * 65)
print('STEP 5 -- Verification')
print('=' * 65)

PASS = True

# ── 1. Original Excel unchanged ───────────────────────────────────────────────
df_orig_chk = pd.read_excel(EXCEL_FILE)
ok = len(df_orig_chk) == 101 and list(df_orig_chk.columns) == orig_cols
PASS = PASS and ok
print(f'  {"[PASS]" if ok else "[FAIL]"}',
      f'Original Excel unchanged  ({len(df_orig_chk)} rows, {len(orig_cols)} cols)')

# ── 2. No BACK in new Excel ───────────────────────────────────────────────────
df_out     = pd.read_excel(EXCEL_OUT)
back_count = int((df_out['Primary Label'] == 'BACK').sum())
ok = back_count == 0
PASS = PASS and ok
print(f'  {"[PASS]" if ok else "[FAIL]"}',
      f'Zero BACK rows in {EXCEL_OUT}  (found: {back_count})')

# ── 3. No BACK in topup feature cache ─────────────────────────────────────────
tc_chk = torch.load(TOPUP_CACHE, weights_only=False)
back_in_cache = int(np.sum(np.array(tc_chk['labels']) == 'BACK'))
ok = back_in_cache == 0
PASS = PASS and ok
print(f'  {"[PASS]" if ok else "[FAIL]"}',
      f'Zero BACK labels in topup cache  (found: {back_in_cache})')

# ── 4. CRC cache untouched ────────────────────────────────────────────────────
crc_chk = torch.load(CRC_CACHE, weights_only=False)
ok = len(crc_chk['labels']) == len(y_crc_full)
PASS = PASS and ok
print(f'  {"[PASS]" if ok else "[FAIL]"}',
      f'CRC cache unchanged  ({len(crc_chk["labels"]):,} features)')

# ── 5. Correct total row count ────────────────────────────────────────────────
n_topup_added = int(needs_df['crc_to_sample'].sum())
back_before   = 0
expected_total = 101 + n_topup_added - back_before
ok = len(df_out) == expected_total
PASS = PASS and ok
print(f'  {"[PASS]" if ok else "[FAIL]"}',
      f'New Excel row count: {len(df_out)} (expected {expected_total})')

# ── 6. Top-up classes hit exactly TARGET in the Excel ─────────────────────────
new_dist = df_out['Primary Label'].value_counts()
topup_ok = all(int(new_dist.get(cls, 0)) == TARGET for cls in TOPUP_CLASSES)
ok = topup_ok
PASS = PASS and ok
if ok:
    print(f'  [PASS] All top-up classes reach exactly {TARGET} in new Excel')
else:
    for cls in TOPUP_CLASSES:
        n = int(new_dist.get(cls, 0))
        mark = '[PASS]' if n == TARGET else '[FAIL]'
        print(f'  {mark} {cls}: {n} (expected {TARGET})')

# ── 7. LYA never merged with LYM; LYM has correct count ──────────────────────
lya_n = int((df_out['Primary Label'] == 'LYA').sum())
lym_n = int((df_out['Primary Label'] == 'LYM').sum())
ok = lym_n == TARGET  # LYM is now a top-up class — expect exactly TARGET rows
PASS = PASS and ok
print(f'  {"[PASS]" if ok else "[FAIL]"}',
      f'LYA rows: {lya_n} | LYM rows: {lym_n} (expected {TARGET} — sampled from CRC-100K)')

# ── 8. No back_patch_ids.csv filenames in new outputs ────────────────────────
back_fname_hits = [p for p in [EXCEL_OUT, TOPUP_CACHE]
                   if 'back_patch_ids' in os.path.basename(p).lower()]
ok = len(back_fname_hits) == 0
PASS = PASS and ok
print(f'  {"[PASS]" if ok else "[FAIL]"}',
      'No back_patch_ids.csv filenames in new pipeline outputs')

# ── FINAL SUMMARY TABLE ───────────────────────────────────────────────────────
print()
print('  FINAL SUMMARY TABLE')
print('  ' + '-' * 57)
print(f'  {"label":<8} {"original":>10} {"crc_topup_added":>16} {"new_total":>10}')
print('  ' + '-' * 57)

topup_added_map = needs_df.set_index('label')['crc_to_sample'].to_dict()
orig_dist       = df_orig_chk['Primary Label'].value_counts()
all_labels      = sorted(df_out['Primary Label'].dropna().unique())

for lbl in all_labels:
    orig_n  = int(orig_dist.get(lbl, 0))
    added_n = int(topup_added_map.get(lbl, 0))
    total_n = int(new_dist.get(lbl, 0))
    print(f'  {lbl:<8} {orig_n:>10} {added_n:>16} {total_n:>10}')

print('  ' + '-' * 57)
print(f'  {"TOTAL":<8} {len(df_orig_chk):>10}',
      f'{n_topup_added:>16} {len(df_out):>10}')
print()
print(f'  {"ALL CHECKS PASSED" if PASS else "SOME CHECKS FAILED -- see [FAIL] lines above"}')

STEP 5 -- Verification
  [PASS] Original Excel unchanged  (101 rows, 10 cols)
  [PASS] Zero BACK rows in fewshot_combined_to20.xlsx  (found: 0)
  [PASS] Zero BACK labels in topup cache  (found: 0)
  [PASS] CRC cache unchanged  (100,000 features)
  [PASS] New Excel row count: 200 (expected 200)
  [PASS] All top-up classes reach exactly 20 in new Excel
  [PASS] LYA rows: 12 | LYM rows: 20 (expected 20 — sampled from CRC-100K)
  [PASS] No back_patch_ids.csv filenames in new pipeline outputs

  FINAL SUMMARY TABLE
  ---------------------------------------------------------
  label      original  crc_topup_added  new_total
  ---------------------------------------------------------
  ADE              11                0         11
  ADI               5               15         20
  CAR              11                0         11
  DEB               5               15         20
  LYA              12                0         12
  LYM               0               20         20
  MES         